In [1]:
import pandas as pd
import os

DATA_PATH = "/kaggle/input/datasets/ditioza/fakenewsproject/"

train_path = os.path.join(DATA_PATH, "multimodal_train.tsv")
val_path   = os.path.join(DATA_PATH, "multimodal_validate.tsv")
test_path  = os.path.join(DATA_PATH, "multimodal_test_public.tsv")

# Read files
train_df = pd.read_csv(train_path, sep='\t')
val_df   = pd.read_csv(val_path, sep='\t')
test_df  = pd.read_csv(test_path, sep='\t')

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

train_df.head()

Train: (564000, 16)
Val: (59342, 16)
Test: (59319, 16)


,author,clean_title,created_utc,domain,hasImage,id,image_url,linked_submission_id,num_comments,score,subreddit,title,upvote_ratio,2_way_label,3_way_label,6_way_label
0,Alexithymia,my walgreens offbrand mucinex was engraved wit...,1.551641e+09,i.imgur.com,True,awxhir,https://external-preview.redd.it/WylDbZrnbvZdB...,NaN,2.0,12,mildlyinteresting,My Walgreens offbrand Mucinex was engraved wit...,0.84,1,0,0
1,VIDCAs17,this concerned sink with a tiny hat,1.534727e+09,i.redd.it,True,98pbid,https://preview.redd.it/wsfx0gp0f5h11.jpg?widt...,NaN,2.0,119,pareidolia,This concerned sink with a tiny hat,0.99,0,2,2
2,prometheus1123,hackers leak emails from uae ambassador to us,1.496511e+09,aljazeera.com,True,6f2cy5,https://external-preview.redd.it/6fNhdbc6K1vFA...,NaN,1.0,44,neutralnews,Hackers leak emails from UAE ambassador to US,0.92,1,0,0
3,NaN,puppy taking in the view,1.471341e+09,i.imgur.com,True,4xypkv,https://external-preview.redd.it/HLtVNhTR6wtYt...,NaN,26.0,250,photoshopbattles,PsBattle: Puppy taking in the view,0.95,1,0,0
4,3rikR3ith,i found a face in my sheet music too,1.525318e+09,i.redd.it,True,8gnet9,https://preview.redd.it/ri7ut2wn8kv01.jpg?widt...,NaN,2.0,13,pareidolia,I found a face in my sheet music too!,0.84,0,2,2


In [2]:
import pandas as pd
import os

# Paths (update dataset name)
DATA_PATH = "/kaggle/input/datasets/ditioza/fakenewsproject/"
OUTPUT_PATH = "/kaggle/working/"

# File paths
train_path = os.path.join(DATA_PATH, "multimodal_train.tsv")
val_path = os.path.join(DATA_PATH, "multimodal_validate.tsv")
test_path = os.path.join(DATA_PATH, "multimodal_test_public.tsv")

# Load datasets
train_df = pd.read_csv(train_path, sep='\t')
val_df = pd.read_csv(val_path, sep='\t')
test_df = pd.read_csv(test_path, sep='\t')

# Sample sizes
train_sample = train_df.sample(n=80000, random_state=42)
val_sample = val_df.sample(n=15000, random_state=42)
test_sample = test_df.sample(n=20000, random_state=42)

# Save new files
train_sample.to_csv(os.path.join(OUTPUT_PATH, "train_sample.tsv"), sep='\t', index=False)
val_sample.to_csv(os.path.join(OUTPUT_PATH, "val_sample.tsv"), sep='\t', index=False)
test_sample.to_csv(os.path.join(OUTPUT_PATH, "test_sample.tsv"), sep='\t', index=False)

print("✅ Sampled files created in /kaggle/working/")

✅ Sampled files created in /kaggle/working/


In [3]:
import os
import pandas as pd
import requests
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Paths
DATA_PATH = "/kaggle/input/datasets/ditioza/sample/"
BASE_DIR = "/kaggle/working/images/"

# Create folders
splits = ["train", "val", "test"]
for split in splits:
    os.makedirs(os.path.join(BASE_DIR, split), exist_ok=True)

# Files mapping
file_map = {
    "train": "train_sample.tsv",
    "val": "val_sample.tsv",
    "test": "test_sample.tsv"
}

IMAGE_COL = "image_url"  # update if needed

# Setup session
session = requests.Session()
retries = Retry(total=3, backoff_factor=0.5)
adapter = HTTPAdapter(max_retries=retries)
session.mount("http://", adapter)
session.mount("https://", adapter)

# Download function
def download_image(split, i, url):
    filename = os.path.join(BASE_DIR, split, f"{i}.jpg")
    
    if os.path.exists(filename):
        return "skipped"
    
    try:
        r = session.get(url, timeout=5)
        if r.status_code == 200 and r.content:
            with open(filename, "wb") as f:
                f.write(r.content)
            return "success"
    except:
        pass
    
    return "failed"

# Process each split
for split, file in file_map.items():
    print(f"\nProcessing {split}...")
    
    df = pd.read_csv(os.path.join(DATA_PATH, file), sep='\t')
    
    results = {"success": 0, "failed": 0, "skipped": 0}
    
    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = [
            executor.submit(download_image, split, i, url)
            for i, url in enumerate(df[IMAGE_COL])
        ]
        
        for future in tqdm(as_completed(futures), total=len(futures)):
            result = future.result()
            results[result] += 1
    
    print(f"{split} results:", results)


Processing train...


100%|██████████| 80000/80000 [1:14:31<00:00, 17.89it/s]


train results: {'success': 52928, 'failed': 27072, 'skipped': 0}

Processing val...


100%|██████████| 15000/15000 [13:50<00:00, 18.07it/s]


val results: {'success': 9990, 'failed': 5010, 'skipped': 0}

Processing test...


100%|██████████| 20000/20000 [18:16<00:00, 18.24it/s]

test results: {'success': 13353, 'failed': 6647, 'skipped': 0}


In [4]:
import os
import pandas as pd

BASE_DIR = "/kaggle/working/images/"
DATA_PATH = "/kaggle/input/datasets/ditioza/sample/"
OUTPUT_PATH = "/kaggle/working/"

file_map = {
    "train": "train_sample.tsv",
    "val": "val_sample.tsv",
    "test": "test_sample.tsv"
}

# ⚠️ UPDATE these column names based on your dataset
TEXT_COL = "text"      # or 'caption' / 'title'
LABEL_COL = "label"    # or 'target'

for split, file in file_map.items():
    print(f"\nProcessing {split}...")
    
    df = pd.read_csv(os.path.join(DATA_PATH, file), sep='\t')
    
    # Keep only required columns
    TEXT_COL = "clean_title"
    LABEL_COL = "2_way_label"

    df = df[[TEXT_COL, LABEL_COL]].copy()
    
    valid_data = []
    
    for i in range(len(df)):
        img_path = os.path.join(BASE_DIR, split, f"{i}.jpg")
        
        if os.path.exists(img_path):
            row = df.iloc[i].to_dict()
            row["image_path"] = f"images/{split}/{i}.jpg"
            valid_data.append(row)
    
    filtered_df = pd.DataFrame(valid_data)
    
    # Basic cleaning
    filtered_df = filtered_df.dropna()
    filtered_df = filtered_df.drop_duplicates()
    
    # Save
    filtered_df.to_csv(
        os.path.join(OUTPUT_PATH, f"{split}_final.tsv"),
        sep='\t',
        index=False
    )
    
    print(f"{split}: {len(filtered_df)} final rows")


Processing train...
train: 52928 final rows

Processing val...
val: 9990 final rows

Processing test...
test: 13353 final rows
